# Notebook 2 — Travel Time Estimation per Route

**Technique:** Statistical aggregation over trip segments + time-of-day stratification

**Goal:** Estimate how long each route takes at different hours of the day, and detect the April 30 military parade impact as a spike in travel time.

**Method:**
1. Load TripSegment records from MinIO (`trips/` prefix)
2. Group by `routeNo` × `hourOfDay`
3. Compute median, P25, P75 trip duration
4. Visualise as a heatmap (route × hour, colour = median duration)
5. Overlay April 30 data vs. 50-day baseline

In [ ]:
import boto3, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timezone
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)
BUCKET = 'bus-history'

In [ ]:
def load_trip_segments(max_objects=2000):
    records = []
    paginator = s3.get_paginator('list_objects_v2')
    count = 0
    for page in paginator.paginate(Bucket=BUCKET, Prefix='trips/'):
        for obj in page.get('Contents', []):
            if count >= max_objects: break
            body = s3.get_object(Bucket=BUCKET, Key=obj['Key'])['Body'].read()
            records.append(json.loads(body))
            count += 1
    return pd.DataFrame(records)

trips = load_trip_segments()
print(f'Loaded {len(trips):,} trip segments')
print(trips.dtypes)

In [ ]:
# Derive hour of day (ICT = UTC+7)
trips['tripStart'] = pd.to_numeric(trips['tripStart'], errors='coerce')
trips['hour'] = trips['tripStart'].apply(
    lambda ts: datetime.fromtimestamp(ts, tz=timezone.utc).hour if pd.notna(ts) else None
)
trips['date'] = trips['tripStart'].apply(
    lambda ts: datetime.fromtimestamp(ts, tz=timezone.utc).strftime('%Y-%m-%d') if pd.notna(ts) else None
)
trips['durationMin'] = pd.to_numeric(trips['durationSeconds'], errors='coerce') / 60

# Filter plausible trips (5 min – 3 hours)
trips_clean = trips[(trips['durationMin'] >= 5) & (trips['durationMin'] <= 180)].copy()
print(f'Plausible trips: {len(trips_clean):,} / {len(trips):,}')

In [ ]:
# ── Travel time heatmap: route × hour ────────────────────────────────────
top_routes = trips_clean['routeNo'].value_counts().head(15).index.tolist()
pivot = trips_clean[trips_clean['routeNo'].isin(top_routes)].pivot_table(
    index='routeNo', columns='hour', values='durationMin', aggfunc='median'
)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot, cmap='RdYlGn_r', annot=False, fmt='.0f',
            cbar_kws={'label': 'Median trip duration (min)'}, ax=ax)
ax.set_title('Travel Time Heatmap — Top 15 Routes by Hour of Day (ICT)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Route')
plt.tight_layout()
plt.savefig('/tmp/travel_time_heatmap.png', dpi=150)
plt.show()
print('Heatmap saved.')

In [ ]:
# ── April 30 impact analysis ─────────────────────────────────────────────
# April 30 2025 = military parade day (HCMC road closures)
apr30 = trips_clean[trips_clean['date'] == '2025-04-30']
baseline = trips_clean[trips_clean['date'] != '2025-04-30']

if len(apr30) > 0:
    comparison = pd.merge(
        baseline.groupby('routeNo')['durationMin'].median().rename('baseline_median'),
        apr30.groupby('routeNo')['durationMin'].median().rename('apr30_median'),
        left_index=True, right_index=True, how='inner'
    )
    comparison['impact_factor'] = comparison['apr30_median'] / comparison['baseline_median']
    comparison = comparison.sort_values('impact_factor', ascending=False)
    print('Routes most affected by April 30 parade:')
    print(comparison.head(10))

    fig, ax = plt.subplots(figsize=(10, 5))
    comparison['impact_factor'].head(15).plot(kind='bar', ax=ax, color='#e94560')
    ax.axhline(1.0, color='green', linestyle='--', label='baseline')
    ax.set_title('Travel Time Impact — April 30 vs Baseline (factor)')
    ax.set_ylabel('Apr30 median / baseline median')
    ax.set_xlabel('Route')
    plt.tight_layout()
    plt.savefig('/tmp/apr30_impact.png', dpi=150)
    plt.show()
else:
    print('April 30 data not yet in MinIO — run the producer past that date first.')